# 02 Data Preparation

Cleans the synthetic dataset, engineers modelling features, and exports processed datasets.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.

# Air Quality Data Preparation for Modelling

This notebook prepares the raw indoor air quality dataset for later modelling.

It covers:
- loading the raw CSV file
- basic data cleaning
- datetime feature creation
- exposure feature calculation
- event indicator creation
- ventilation and source-related feature engineering
- creation of a model-ready encoded dataset
- exporting processed CSV files for modelling


In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Input/output paths
# ------------------------------------------------------------
RAW_DATA_PATH = DATA_DIR / "synthetic_air_quality_dataset_final.csv"
OUTPUT_DIR = PROCESSED_DATA_DIR

CLEAN_FEATURED_PATH = OUTPUT_DIR / "air_quality_cleaned_featured.csv"
MODEL_READY_PATH = OUTPUT_DIR / "air_quality_model_ready_encoded.csv"
FEATURE_INFO_PATH = OUTPUT_DIR / "feature_lists.json"

print("Input dataset:", RAW_DATA_PATH)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 1. Load Raw Dataset
# ============================================================

df_raw = pd.read_csv(RAW_DATA_PATH)

print("Raw data shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())

In [ ]:
# ============================================================
# 2. Basic Data Cleaning
# ============================================================

df = df_raw.copy()

# ------------------------------------------------------------
# 2.1 Standardise column names
# ------------------------------------------------------------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# ------------------------------------------------------------
# 2.2 Parse datetime columns
# ------------------------------------------------------------
datetime_cols = ["day", "start_time_local", "end_time_local"]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# ------------------------------------------------------------
# 2.3 Clean categorical columns
# ------------------------------------------------------------
categorical_cols = [
    "time_of_day",
    "motion_type",
    "environment_type",
    "activity_type",
    "cooking_method",
    "appliance_type"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .fillna("unknown")
        )

# ------------------------------------------------------------
# 2.4 Convert boolean / binary columns to integers
# ------------------------------------------------------------
binary_cols = [
    "cooking_event",
    "gas_cooking_or_gas_hob",
    "wood_burning_event",
    "smoking_vaping_exposure",
    "window_open",
    "extractor_fan_on"
]

for col in binary_cols:
    if col in df.columns:
        # Handles True/False, 1/0, and string versions
        df[col] = df[col].replace({
            True: 1, False: 0,
            "true": 1, "false": 0,
            "True": 1, "False": 0,
            "yes": 1, "no": 0,
            "Yes": 1, "No": 0
        })
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------------------
# 2.5 Handle numeric missing values and impossible values
# ------------------------------------------------------------
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Non-negative columns
non_negative_cols = [
    "duration_min",
    "outdoor_pm25", "outdoor_pm10", "outdoor_no2", "outdoor_o3", "outdoor_so2",
    "cooking_duration_min",
    "pm25_rate", "pm10_rate", "no2_rate"
]

for col in non_negative_cols:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

# ------------------------------------------------------------
# 2.6 Fill missing time_of_day using hour if needed
# ------------------------------------------------------------
if "start_time_local" in df.columns:
    df["hour"] = df["start_time_local"].dt.hour

    def infer_time_of_day(hour):
        if pd.isna(hour):
            return "unknown"
        if 5 <= hour < 12:
            return "morning"
        elif 12 <= hour < 17:
            return "afternoon"
        elif 17 <= hour < 21:
            return "evening"
        else:
            return "night"

    if "time_of_day" in df.columns:
        missing_mask = df["time_of_day"].isin(["unknown", "<na>", "nan"])
        df.loc[missing_mask, "time_of_day"] = df.loc[missing_mask, "hour"].apply(infer_time_of_day)

print("Cleaned data shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
# ============================================================
# 3. Feature Engineering
# ============================================================

# ------------------------------------------------------------
# 3.1 Time-based features
# ------------------------------------------------------------
if "start_time_local" in df.columns:
    df["hour"] = df["start_time_local"].dt.hour
    df["day_of_week"] = df["start_time_local"].dt.day_name().str.lower()
    df["day_of_week_num"] = df["start_time_local"].dt.dayofweek
    df["month"] = df["start_time_local"].dt.month
    df["is_weekend"] = df["day_of_week_num"].isin([5, 6]).astype(int)

if "duration_min" in df.columns:
    df["duration_hours"] = df["duration_min"] / 60

# ------------------------------------------------------------
# 3.2 Exposure features
# Exposure = pollutant rate × duration
# ------------------------------------------------------------
target_cols = ["pm25_rate", "pm10_rate", "no2_rate"]

for pollutant in ["pm25", "pm10", "no2"]:
    rate_col = f"{pollutant}_rate"
    exposure_col = f"{pollutant}_exposure"

    if rate_col in df.columns and "duration_min" in df.columns:
        df[exposure_col] = df[rate_col] * df["duration_min"]
        df[f"log1p_{rate_col}"] = np.log1p(df[rate_col])
        df[f"log1p_{exposure_col}"] = np.log1p(df[exposure_col])

# ------------------------------------------------------------
# 3.3 Pollution event indicators
# These are useful for two-part / hurdle modelling.
# ------------------------------------------------------------
for target in target_cols:
    if target in df.columns:
        df[f"{target}_event"] = (df[target] > 0).astype(int)

        # High event based on upper quartile of positive values.
        positive_values = df.loc[df[target] > 0, target]
        if len(positive_values) > 0:
            threshold = positive_values.quantile(0.75)
            df[f"{target}_high_event"] = (df[target] >= threshold).astype(int)
        else:
            df[f"{target}_high_event"] = 0

# ------------------------------------------------------------
# 3.4 Behaviour and source features
# ------------------------------------------------------------
if {"window_open", "extractor_fan_on"}.issubset(df.columns):
    df["ventilation_count"] = df["window_open"] + df["extractor_fan_on"]
    df["any_ventilation"] = (df["ventilation_count"] > 0).astype(int)

source_cols = [
    col for col in [
        "gas_cooking_or_gas_hob",
        "wood_burning_event",
        "smoking_vaping_exposure"
    ] if col in df.columns
]

if source_cols:
    df["combustion_source_count"] = df[source_cols].sum(axis=1)
    df["any_combustion_source"] = (df["combustion_source_count"] > 0).astype(int)

if {"cooking_event", "cooking_duration_min"}.issubset(df.columns):
    df["cooking_intensity"] = df["cooking_event"] * df["cooking_duration_min"]

if {"cooking_event", "any_ventilation"}.issubset(df.columns):
    df["cooking_without_ventilation"] = (
        (df["cooking_event"] == 1) & (df["any_ventilation"] == 0)
    ).astype(int)

if {"gas_cooking_or_gas_hob", "any_ventilation"}.issubset(df.columns):
    df["gas_cooking_without_ventilation"] = (
        (df["gas_cooking_or_gas_hob"] == 1) & (df["any_ventilation"] == 0)
    ).astype(int)

# ------------------------------------------------------------
# 3.5 Outdoor pollution context features
# ------------------------------------------------------------
outdoor_cols = ["outdoor_pm25", "outdoor_pm10", "outdoor_no2", "outdoor_o3", "outdoor_so2"]

for col in outdoor_cols:
    if col in df.columns:
        q75 = df[col].quantile(0.75)
        df[f"{col}_high"] = (df[col] >= q75).astype(int)

print("Feature-engineered data shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# 4. Save Cleaned + Feature-Engineered Dataset
# ============================================================

df.to_csv(CLEAN_FEATURED_PATH, index=False)

print("Saved cleaned + featured dataset to:")
print(CLEAN_FEATURED_PATH)

print("Shape:", df.shape)

In [ ]:
# ============================================================
# 5. Create Model-Ready Encoded Dataset
# ============================================================

# Targets and derived outcome columns that should not be used as predictors
target_and_outcome_cols = [
    "pm25_rate", "pm10_rate", "no2_rate",
    "pm25_exposure", "pm10_exposure", "no2_exposure",
    "log1p_pm25_rate", "log1p_pm10_rate", "log1p_no2_rate",
    "log1p_pm25_exposure", "log1p_pm10_exposure", "log1p_no2_exposure",
    "pm25_rate_event", "pm10_rate_event", "no2_rate_event",
    "pm25_rate_high_event", "pm10_rate_high_event", "no2_rate_high_event"
]

# Columns not directly useful for model input
drop_from_features = [
    "day", "start_time_local", "end_time_local"
]

# Candidate feature columns
feature_cols = [
    col for col in df.columns
    if col not in target_and_outcome_cols and col not in drop_from_features
]

# Identify categorical and numeric features
categorical_features = df[feature_cols].select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_features = [col for col in feature_cols if col not in categorical_features]

print("Number of feature columns before encoding:", len(feature_cols))
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

# One-hot encode categorical features
df_features = pd.get_dummies(
    df[feature_cols],
    columns=categorical_features,
    drop_first=False,
    dummy_na=False,
    dtype=int
)

# Add targets back for modelling
model_ready = pd.concat(
    [
        df_features,
        df[[col for col in target_and_outcome_cols if col in df.columns]]
    ],
    axis=1
)

# Save model-ready encoded dataset
model_ready.to_csv(MODEL_READY_PATH, index=False)

# Save feature information
feature_info = {
    "raw_data_path": RAW_DATA_PATH,
    "clean_featured_path": CLEAN_FEATURED_PATH,
    "model_ready_path": MODEL_READY_PATH,
    "feature_columns_before_encoding": feature_cols,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "encoded_feature_columns": df_features.columns.tolist(),
    "target_and_outcome_columns": [col for col in target_and_outcome_cols if col in df.columns]
}

with open(FEATURE_INFO_PATH, "w") as f:
    json.dump(feature_info, f, indent=4, default=str)

print("Saved model-ready encoded dataset to:")
print(MODEL_READY_PATH)
print("Shape:", model_ready.shape)

print("\nSaved feature information to:")
print(FEATURE_INFO_PATH)

display(model_ready.head())

In [ ]:
# ============================================================
# 6. Final Checks
# ============================================================

print("Cleaned + featured dataset:")
print(df.shape)

print("\nModel-ready encoded dataset:")
print(model_ready.shape)

print("\nMissing values in cleaned dataset:")
print(df.isna().sum().sum())

print("\nMissing values in model-ready dataset:")
print(model_ready.isna().sum().sum())

print("\nFiles created:")
for file in [CLEAN_FEATURED_PATH, MODEL_READY_PATH, FEATURE_INFO_PATH]:
    print(file, "->", os.path.exists(file))